In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
from torchvision import models
from google.colab import drive

# ✅ Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# ✅ Model Save Path
model_save_path = "/content/drive/MyDrive/best_pest_model.pth"

# ✅ Dataset Path
data_dir = "/content/drive/MyDrive/image_dataset"

# ✅ Enable GPU Optimization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# ✅ Function to Load Data
def get_dataloaders(data_dir, batch_size=64, test_split=0.2):
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
        transforms.GaussianBlur(kernel_size=(5, 5)),
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
        transforms.RandomErasing(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = datasets.ImageFolder(root=data_dir)
    num_classes = len(dataset.classes)
    print("Class-to-Index Mapping:", dataset.class_to_idx)

    train_size = int((1 - test_split) * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_dataset.dataset.transform = train_transform
    val_dataset.dataset.transform = val_transform

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, dataset.classes, num_classes

# ✅ Load Dataset
train_loader, val_loader, class_names, num_classes = get_dataloaders(data_dir)

# ✅ Load Pretrained Model (ResNet-34)
model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.to(device)

# ✅ Define Loss, Optimizer & Scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

# ✅ Training with Early Stopping & Gradient Accumulation
best_val_loss = float("inf")
early_stopping_counter = 0
patience = 5
grad_accumulation = 2  # Accumulate gradients every 2 batches

for epoch in range(30):
    model.train()
    train_loss = 0
    optimizer.zero_grad()

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        if (batch_idx + 1) % grad_accumulation == 0:
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item()

    val_loss = 0
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    print(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

    # ✅ Save Best Model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), model_save_path)
        print(f"✅ Model saved successfully at {model_save_path}")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= patience:
            print("⚠️ Early stopping triggered. Training stopped.")
            break

    scheduler.step()



Mounted at /content/drive
Class-to-Index Mapping: {'Brown Plant Hopper': 0, 'Chickpea pod borers': 1, 'Coffee borer': 2, 'Diamondback Moth': 3, 'Helicoverpa Armigera': 4, 'Leafhoppers': 5, 'Leafminer': 6, 'Mango Leafhopper': 7, 'Mealybug': 8, 'Red Cotton Bug': 9, 'Rice Hispa': 10, 'Spotted Pod borer': 11, 'Sugarcane top borer': 12, 'Thrips': 13, 'White Grubs': 14, 'Yellow Rice Borer': 15}


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|██████████| 83.3M/83.3M [00:01<00:00, 43.9MB/s]


Epoch 1 - Train Loss: 1.2864 | Val Loss: 1.0667
✅ Model saved successfully at /content/drive/MyDrive/best_pest_model.pth
Epoch 2 - Train Loss: 0.5970 | Val Loss: 0.8250
✅ Model saved successfully at /content/drive/MyDrive/best_pest_model.pth
Epoch 3 - Train Loss: 0.4287 | Val Loss: 0.7265
✅ Model saved successfully at /content/drive/MyDrive/best_pest_model.pth
Epoch 4 - Train Loss: 0.3693 | Val Loss: 0.6800
✅ Model saved successfully at /content/drive/MyDrive/best_pest_model.pth
Epoch 5 - Train Loss: 0.3540 | Val Loss: 0.6701
✅ Model saved successfully at /content/drive/MyDrive/best_pest_model.pth
Epoch 6 - Train Loss: 0.3510 | Val Loss: 0.6613
✅ Model saved successfully at /content/drive/MyDrive/best_pest_model.pth
Epoch 7 - Train Loss: 0.3415 | Val Loss: 0.6501
✅ Model saved successfully at /content/drive/MyDrive/best_pest_model.pth
Epoch 8 - Train Loss: 0.3350 | Val Loss: 0.6490
✅ Model saved successfully at /content/drive/MyDrive/best_pest_model.pth
Epoch 9 - Train Loss: 0.3312 | V